# Dashboard access tracker walkthrough

## Project purpose

This notebook documents the workflow for a SQL Server based dashboard access tracker. The project replaces spreadsheet based tracking with a relational database structure, stored procedures, views, and future Python automation.

## What this notebook demonstrates

- How the database is structured
- How access requests are created
- How users are connected to clients
- How stored procedures support automation
- How Python can query SQL Server for reporting and workflow validation

In [1]:
import pyodbc
import pandas as pd

print("Environment connected successfully")

Environment connected successfully


## SQL Server connection

The following section connects Python to SQL Server using pyodbc. This allows the notebook to query reporting views directly from the relational database.

In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.db_connection import get_connection

conn = get_connection()

Database connection established successfully.


## Query reporting views

Now that Python is connected to SQL Server, the notebook can query reporting views and display the results with pandas.

In [ ]:
query = """
SELECT *
FROM dbo.vw_AccessRequests;
"""

df_access_requests = pd.read_sql(query, conn)

df_access_requests

C:\Users\Christian.Arroyo\AppData\Local\Temp\ipykernel_15996\538734020.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_access_requests = pd.read_sql(query, conn)


,AccessRequestID,DashboardUserID,FirstName,LastName,Email,ContractManagerID,ContractManager,ContractManagerEmail,RequestType,RequestStatus,RequestedBy,Notes,DateRequested,DateEmailSent,DateCompleted
0,1,1,Jane,Doe,jane.doe@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Draft,Christian Arroyo,Initial dashboard access request for Jane Doe,2026-05-13 15:44:25.343,NaT,NaT
1,2,2,John,Smith,john.smith@example.com,2,Michael Rivera,michael.rivera@example.com,Add,Email Sent,Christian Arroyo,Access request submitted to third-party support,2026-05-13 15:44:25.343,NaT,NaT
2,3,3,Emily,Davis,emily.davis@example.com,3,Ashley Turner,ashley.turner@example.com,Modify,Completed,Christian Arroyo,Additional client access granted,2026-05-13 15:44:25.343,NaT,NaT
3,4,1,Jane,Doe,jane.doe@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Draft,Christian Arroyo,Testing stored procedure workflow,2026-05-13 16:11:15.647,NaT,NaT
4,5,1,Jane,Doe,jane.doe@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Draft,Christian Arroyo,Testing stored procedure workflow,2026-05-13 16:17:46.260,NaT,NaT
5,6,4,Robert,Miller,robert.miller@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Completed,Christian Arroyo,Testing stored procedure workflow,2026-05-13 16:23:25.727,2026-05-13 16:23:25.840,2026-05-13 16:24:24.150


In [ ]:
query = """
SELECT *
FROM dbo.vw_UserClientAccess;
"""

df_user_client_access = pd.read_sql(query, conn)

df_user_client_access

C:\Users\Christian.Arroyo\AppData\Local\Temp\ipykernel_15996\3706654145.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_user_client_access = pd.read_sql(query, conn)


,UserClientAccessID,DashboardUserID,FirstName,LastName,Email,ClientID,ClientCode,ClientName,IsActive,DateAccessRequested,DateAccessConfirmed,DateAccessRemoved
0,1,1,Jane,Doe,jane.doe@example.com,1,CLIENT001,Sample Client One,True,2026-05-13 15:44:25.353,2026-05-13 15:44:25.353,None
1,2,1,Jane,Doe,jane.doe@example.com,2,CLIENT002,Sample Client Two,True,2026-05-13 15:44:25.353,2026-05-13 15:44:25.353,None
2,3,2,John,Smith,john.smith@example.com,2,CLIENT002,Sample Client Two,True,2026-05-13 15:44:25.353,2026-05-13 15:44:25.353,None
3,4,2,John,Smith,john.smith@example.com,3,CLIENT003,Sample Client Three,True,2026-05-13 15:44:25.353,2026-05-13 15:44:25.353,None
4,5,3,Emily,Davis,emily.davis@example.com,1,CLIENT001,Sample Client One,True,2026-05-13 15:44:25.353,2026-05-13 15:44:25.353,None
5,6,3,Emily,Davis,emily.davis@example.com,4,CLIENT004,Sample Client Four,True,2026-05-13 15:44:25.353,2026-05-13 15:44:25.353,None


## Basic access reporting

These queries show how the relational model can answer common business questions, such as how many users have access to each client and which requests are still open.

In [ ]:
query = """
SELECT
    ClientCode,
    ClientName,
    COUNT(*) AS ActiveUserCount
FROM dbo.vw_UserClientAccess
WHERE IsActive = 1
GROUP BY
    ClientCode,
    ClientName
ORDER BY
    ActiveUserCount DESC;
"""

df_active_users_by_client = pd.read_sql(query, conn)

df_active_users_by_client

C:\Users\Christian.Arroyo\AppData\Local\Temp\ipykernel_15996\2736208846.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_active_users_by_client = pd.read_sql(query, conn)


,ClientCode,ClientName,ActiveUserCount
0,CLIENT001,Sample Client One,2
1,CLIENT002,Sample Client Two,2
2,CLIENT003,Sample Client Three,1
3,CLIENT004,Sample Client Four,1


In [ ]:
query = """
SELECT *
FROM dbo.vw_AccessRequests
WHERE DateCompleted IS NULL
ORDER BY DateRequested DESC;
"""

df_open_requests = pd.read_sql(query, conn)

df_open_requests

C:\Users\Christian.Arroyo\AppData\Local\Temp\ipykernel_15996\355573803.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_open_requests = pd.read_sql(query, conn)


,AccessRequestID,DashboardUserID,FirstName,LastName,Email,ContractManagerID,ContractManager,ContractManagerEmail,RequestType,RequestStatus,RequestedBy,Notes,DateRequested,DateEmailSent,DateCompleted
0,5,1,Jane,Doe,jane.doe@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Draft,Christian Arroyo,Testing stored procedure workflow,2026-05-13 16:17:46.260,None,None
1,4,1,Jane,Doe,jane.doe@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Draft,Christian Arroyo,Testing stored procedure workflow,2026-05-13 16:11:15.647,None,None
2,1,1,Jane,Doe,jane.doe@example.com,1,Sarah Johnson,sarah.johnson@example.com,Add,Draft,Christian Arroyo,Initial dashboard access request for Jane Doe,2026-05-13 15:44:25.343,None,None
3,2,2,John,Smith,john.smith@example.com,2,Michael Rivera,michael.rivera@example.com,Add,Email Sent,Christian Arroyo,Access request submitted to third-party support,2026-05-13 15:44:25.343,None,None
4,3,3,Emily,Davis,emily.davis@example.com,3,Ashley Turner,ashley.turner@example.com,Modify,Completed,Christian Arroyo,Additional client access granted,2026-05-13 15:44:25.343,None,None


## Executing stored procedures from Python

The notebook can also execute SQL Server stored procedures directly. This supports future workflow automation and GUI development.

In [ ]:
cursor = conn.cursor()

new_dashboard_user_id = cursor.execute("""
DECLARE @NewDashboardUserID INT;

EXEC dbo.usp_CreateDashboardUser
    @FirstName = 'Michael',
    @LastName = 'Jordan',
    @Email = 'michael.jordan@example.com',
    @DashboardUserID = @NewDashboardUserID OUTPUT;

SELECT @NewDashboardUserID AS DashboardUserID;
""").fetchone()[0]

conn.commit()

print(f"New DashboardUserID: {new_dashboard_user_id}")

query = f"""
SELECT *
FROM dbo.DashboardUsers
WHERE DashboardUserID = {new_dashboard_user_id};
"""

df_new_user = pd.read_sql(query, conn)

df_new_user

New DashboardUserID: 5


C:\Users\Christian.Arroyo\AppData\Local\Temp\ipykernel_15996\3424744467.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_new_user = pd.read_sql(query, conn)


,DashboardUserID,FirstName,LastName,Email,IsActive,DateCreated,DateModified
0,5,Michael,Jordan,michael.jordan@example.com,True,2026-05-13 18:01:13.903,None
